# Run O: PIDNet-S, binary defect segmentation

PIDNet was published at CVPR 2023: *PIDNet: A Real-time Semantic Segmentation Network Inspired from PID Controller* (arXiv:2206.02066). It's a three-branch network -- P (detail/proportional), I (context/integral), D (boundary/derivative) -- where the D branch predicts object boundaries and actively gates how the P and I branches get fused, mimicking a PID controller's overshoot correction. The official repo (`XuJiacong/PIDNet`) is pure PyTorch with no framework dependency, so `models/pidnet.py` + `models/model_utils.py` are vendored here essentially verbatim (only the PIDNet-S-relevant branch of a couple of `if m==2` checks was kept). Verified locally before ever touching Kaggle: dummy forward+backward gives correct shapes, 100% gradient coverage across all three branches, and the total parameter count (7.72M) matches the paper's reported 7.7M for PIDNet-S almost exactly.

**Faithful 3-part loss**, ported from the official `utils/criterion.py` + `utils/utils.py`'s `FullModel.forward`: `OhemCrossEntropy` (weights [0.4, 1.0]) on the P-branch auxiliary head and the final fused output, plus `BoundaryLoss` (20x class-balanced weighted BCE) on the D-branch's boundary prediction, plus a boundary-masked auxiliary OHEM term that only supervises the final output strongly where the model's own predicted boundary confidence exceeds 0.8 (elsewhere the label is masked to `ignore_label`) -- this is the actual mechanism that makes PIDNet's boundary branch matter, not just an auxiliary target. Boundary ground truth is generated per-sample exactly like the official `datasets/base_dataset.py`'s `gen_sample`: `cv2.Canny(label, 0.1, 0.2)` on the label map, then dilated with a 4x4 kernel. Note: `augment=True` is fixed at construction (not tied to `.train()`/`.eval()`), so the raw model always returns `[aux_p, final, aux_d]` at 1/8 resolution -- upsampling to label resolution happens in the training/eval code, exactly where the official `FullModel.forward` does it, not inside the model itself.

**Pretrained weights: verified unavailable, not silently skipped.** The official repo's README states outright that the author's Google Drive account is gone ("It appears the download links below are no longer working due to my missing Google Drive account"). Checked directly rather than taking that at face value: both an individual weight link and the author's own posted fallback consolidated folder both return Google Drive's "file/folder does not exist" page over a real HTTP request. A HuggingFace listing under `qualcomm/PidNet` turned up in search but only hosts a model card + `release_assets.json` pointing at Qualcomm's own AI Hub asset pipeline, not a plain downloadable checkpoint, so it wasn't usable as a faithful drop-in. This model therefore trains from scratch, using the paper's own from-scratch Kaiming initialization scheme (already built into the PIDNet class's `__init__`) -- clearly logged, `pretrained_backbone_loaded=False` recorded in every output artifact.

Uses the same fixed size-stratified split, W&B logging, and D-FINE-matching summary.csv schema as every other baseline in this project.

Kaggle setup: attach **SmallDefectPreprocessing** as an input, enable Internet, and use a GPU.

In [ ]:
from pathlib import Path
from collections import Counter, defaultdict
import json
import os
import random
import shutil
import subprocess
import sys
import time

RUN_NAME = 'RunO_pidnet_s_imgsz640'
MODEL_LABEL = 'PIDNet-S'
WANDB_PROJECT = 'smallDefectDetection'
WANDB_RUN_NAME = f'{MODEL_LABEL}_segmentation'
IMG_SIZE = 640
BATCH_SIZE = 8
MAX_EPOCHS = 50
PATIENCE = 15
LEARNING_RATE = 6e-5
WEIGHT_DECAY = 1e-2
NUM_WORKERS = 2
SEED = 42

DATASET_NAMES = ['DAGM', 'GC10-DET', 'KolektorSDD2', 'MPDD', 'MTD', 'Severstal', 'VisA']
SIZE_BUCKETS = ['small', 'medium', 'large']
IMAGE_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff'}

KAGGLE_INPUT_ROOT = Path('/kaggle/input')
WORKING_ROOT = Path('/kaggle/working')
RUN_DIR = WORKING_ROOT / 'pidnet_runs' / RUN_NAME
FINAL_OUTPUT_DIR = WORKING_ROOT / 'final_outputs' / RUN_NAME
# Created unconditionally, right here -- not contingent on any later cell's control flow
# (a downloaded-checkpoint try/except, an early-stopping branch, etc.) reaching a line that
# happens to also create it.
RUN_DIR.mkdir(parents=True, exist_ok=True)

print({'run': RUN_NAME, 'model': MODEL_LABEL, 'imgsz': IMG_SIZE, 'batch': BATCH_SIZE, 'max_epochs': MAX_EPOCHS, 'patience': PATIENCE})

In [ ]:
# Kaggle Internet must be enabled for this cell.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'wandb'], check=True)

import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import wandb
from PIL import Image
from torch.optim import AdamW
from torch.utils.data import DataLoader, Dataset

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if DEVICE.type != 'cuda':
    raise RuntimeError('Enable a Kaggle GPU before training.')

torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
np.random.seed(SEED)
random.seed(SEED)
torch.backends.cudnn.benchmark = True
print('Using device:', torch.cuda.get_device_name(0))


def wandb_login_anywhere():
    # Works on RunPod (env var) and Kaggle (Secrets add-on) without ever hardcoding the key.
    api_key = os.environ.get('WANDB_API_KEY')
    if not api_key:
        try:
            from kaggle_secrets import UserSecretsClient
            api_key = UserSecretsClient().get_secret('WANDB_API_KEY')
        except Exception:
            api_key = None
    if api_key:
        wandb.login(key=api_key)
    else:
        wandb.login()


wandb_login_anywhere()
wandb.init(
    project=WANDB_PROJECT,
    name=WANDB_RUN_NAME,
    config={
        'model': MODEL_LABEL,
        'img_size': IMG_SIZE,
        'batch_size': BATCH_SIZE,
        'max_epochs': MAX_EPOCHS,
        'patience': PATIENCE,
        'learning_rate': LEARNING_RATE,
        'weight_decay': WEIGHT_DECAY,
        'seed': SEED,
    },
)

In [ ]:
# Find the normal SmallDefectPreprocessing Kaggle input automatically.
expected_datasets = set(DATASET_NAMES)
source_candidates = []

for root, dirs, _ in os.walk(KAGGLE_INPUT_ROOT):
    matches = expected_datasets.intersection(dirs)
    if len(matches) >= 5:
        source_candidates.append((len(matches), Path(root)))

if not source_candidates:
    raise FileNotFoundError(
        'Could not find the processed dataset. Attach SmallDefectPreprocessing as a Kaggle input.'
    )

source_candidates.sort(key=lambda item: (-item[0], len(str(item[1]))))
SOURCE_ROOT = source_candidates[0][1]
print('Using processed source:', SOURCE_ROOT)
print('Datasets:', sorted(path.name for path in SOURCE_ROOT.iterdir() if path.is_dir()))

In [ ]:
def index_files(directory, suffixes):
    return {
        path.stem: path
        for path in directory.iterdir()
        if path.is_file() and path.suffix.lower() in suffixes
    }

def candidate_stems(image_stem, target):
    base = image_stem.removesuffix('_defect')
    if target == 'mask':
        return [image_stem, image_stem.replace('_defect', '_mask'), base, base + '_mask', base + '_gt']
    return [image_stem, image_stem.replace('_defect', '_bbs'), base, base + '_bbs']

samples = []
missing = []

for dataset_name in DATASET_NAMES:
    for size_bucket in SIZE_BUCKETS:
        bucket_root = SOURCE_ROOT / dataset_name / size_bucket
        image_dir = bucket_root / 'images'
        mask_dir = bucket_root / 'masks'
        label_dir = bucket_root / 'labels_yolo'

        if not image_dir.exists() or not mask_dir.exists() or not label_dir.exists():
            missing.append((dataset_name, size_bucket, 'missing directory'))
            continue

        mask_index = index_files(mask_dir, IMAGE_EXTS)
        label_index = index_files(label_dir, {'.txt'})
        matched = 0

        for image_path in image_dir.iterdir():
            if image_path.suffix.lower() not in IMAGE_EXTS:
                continue

            mask_path = next((mask_index[stem] for stem in candidate_stems(image_path.stem, 'mask') if stem in mask_index), None)
            label_path = next((label_index[stem] for stem in candidate_stems(image_path.stem, 'box') if stem in label_index), None)

            if mask_path is None or label_path is None:
                missing.append((dataset_name, size_bucket, image_path.name))
                continue

            samples.append({
                'image_path': image_path,
                'mask_path': mask_path,
                'label_path': label_path,
                'dataset': dataset_name,
                'size': size_bucket,
                'stratum': dataset_name + '_' + size_bucket,
            })
            matched += 1

        print(f'{dataset_name}/{size_bucket}: {matched} matched')

print('Total matched:', len(samples))
print('Missing:', len(missing))
if len(samples) != 12670:
    raise RuntimeError(f'Expected 12,670 image-mask-label triplets, found {len(samples)}. First missing: {missing[:10]}')

In [ ]:
# Same fixed 70/15/15 stratified split as the other runs.
by_stratum = defaultdict(list)
for sample in samples:
    by_stratum[sample['stratum']].append(sample)

rng = random.Random(SEED)
train_samples, val_samples, test_samples = [], [], []
for _, group in sorted(by_stratum.items()):
    group = list(group)
    rng.shuffle(group)
    train_end = int(len(group) * 0.70)
    val_end = train_end + int(len(group) * 0.15)
    train_samples.extend(group[:train_end])
    val_samples.extend(group[train_end:val_end])
    test_samples.extend(group[val_end:])

rng.shuffle(train_samples)
rng.shuffle(val_samples)
rng.shuffle(test_samples)

test_sets = {
    'overall': test_samples,
    'small': [sample for sample in test_samples if sample['size'] == 'small'],
    'medium': [sample for sample in test_samples if sample['size'] == 'medium'],
    'large': [sample for sample in test_samples if sample['size'] == 'large'],
}

def print_counts(name, split):
    counts = Counter(sample['size'] for sample in split)
    print(f'{name}: total={len(split)}, small={counts["small"]}, medium={counts["medium"]}, large={counts["large"]}')

print_counts('Train', train_samples)
print_counts('Validation', val_samples)
for name, split in test_sets.items():
    print_counts('Test ' + name, split)

assert len(train_samples) == 8858
assert len(val_samples) == 1892
assert len(test_samples) == 1920

In [ ]:
IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
IMAGENET_STD = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)


def load_binary_mask(mask_path, target_size):
    mask = Image.open(mask_path).convert('L')
    if mask.size != target_size:
        mask = mask.resize(target_size, Image.Resampling.NEAREST)
    return (np.asarray(mask) > 0).astype(np.uint8)


def generate_edge(label, edge_size=4, y_k_size=6, x_k_size=6):
    """Boundary ground truth, verbatim from the official datasets/base_dataset.py's gen_sample:
    Canny on the label map itself (class-index discontinuities are the "edges" to detect,
    not natural-image edges -- hence the tiny 0.1/0.2 thresholds), then dilated to a band."""
    edge = cv2.Canny(label, 0.1, 0.2)
    kernel = np.ones((edge_size, edge_size), np.uint8)
    edge = edge[y_k_size:-y_k_size, x_k_size:-x_k_size]
    edge = np.pad(edge, ((y_k_size, y_k_size), (x_k_size, x_k_size)), mode='constant')
    edge = (cv2.dilate(edge, kernel, iterations=1) > 50) * 1.0
    return edge.astype(np.float32)


class DefectMaskDataset(Dataset):
    def __init__(self, split_samples):
        self.samples = split_samples

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        sample = self.samples[index]
        image = Image.open(sample['image_path']).convert('RGB').resize((IMG_SIZE, IMG_SIZE), Image.Resampling.BILINEAR)
        mask = load_binary_mask(sample['mask_path'], (IMG_SIZE, IMG_SIZE))

        pixel_values = torch.from_numpy(np.asarray(image)).permute(2, 0, 1).float() / 255.0
        pixel_values = (pixel_values - IMAGENET_MEAN) / IMAGENET_STD
        labels = torch.from_numpy(mask).long()
        bd_gt = torch.from_numpy(generate_edge(mask))

        return {'pixel_values': pixel_values, 'labels': labels, 'bd_gt': bd_gt}


def make_loader(split_samples, shuffle=False):
    return DataLoader(
        DefectMaskDataset(split_samples),
        batch_size=BATCH_SIZE,
        shuffle=shuffle,
        num_workers=NUM_WORKERS,
        pin_memory=True,
        persistent_workers=NUM_WORKERS > 0,
    )


train_loader = make_loader(train_samples, shuffle=True)
val_loader = make_loader(val_samples)
print('Train batches:', len(train_loader), 'Validation batches:', len(val_loader))

In [ ]:
# ---- Vendored from the official PIDNet repo (XuJiacong/PIDNet): models/pidnet.py + ----
# ---- models/model_utils.py, essentially verbatim (pure PyTorch, no framework dependency). ----
# ---- Only the PIDNet-S-relevant branch of the couple of "if m == 2" checks is kept. ----
# ---- Loss classes ported from utils/criterion.py; the 3-part loss combination from ----
# ---- utils/utils.py's FullModel.forward. ----

BatchNorm2d = nn.BatchNorm2d
bn_mom = 0.1
algc = False


class BasicBlock(nn.Module):
    expansion = 1

    def __init__(self, inplanes, planes, stride=1, downsample=None, no_relu=False):
        super().__init__()
        self.conv1 = nn.Conv2d(inplanes, planes, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = BatchNorm2d(planes, momentum=bn_mom)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv2d(planes, planes, kernel_size=3, padding=1, bias=False)
        self.bn2 = BatchNorm2d(planes, momentum=bn_mom)
        self.downsample = downsample
        self.stride = stride
        self.no_relu = no_relu

    def forward(self, x):
        residual = x
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        if self.downsample is not None:
            residual = self.downsample(x)
        out += residual
        return out if self.no_relu else self.relu(out)


class Bottleneck(nn.Module):
    expansion = 2

    def __init__(self, inplanes, planes, stride=1, downsample=None, no_relu=True):
        super().__init__()
        self.conv1 = nn.Conv2d(inplanes, planes, kernel_size=1, bias=False)
        self.bn1 = BatchNorm2d(planes, momentum=bn_mom)
        self.conv2 = nn.Conv2d(planes, planes, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn2 = BatchNorm2d(planes, momentum=bn_mom)
        self.conv3 = nn.Conv2d(planes, planes * self.expansion, kernel_size=1, bias=False)
        self.bn3 = BatchNorm2d(planes * self.expansion, momentum=bn_mom)
        self.relu = nn.ReLU(inplace=True)
        self.downsample = downsample
        self.stride = stride
        self.no_relu = no_relu

    def forward(self, x):
        residual = x
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.relu(self.bn2(self.conv2(out)))
        out = self.bn3(self.conv3(out))
        if self.downsample is not None:
            residual = self.downsample(x)
        out += residual
        return out if self.no_relu else self.relu(out)


class segmenthead(nn.Module):
    def __init__(self, inplanes, interplanes, outplanes, scale_factor=None):
        super().__init__()
        self.bn1 = BatchNorm2d(inplanes, momentum=bn_mom)
        self.conv1 = nn.Conv2d(inplanes, interplanes, kernel_size=3, padding=1, bias=False)
        self.bn2 = BatchNorm2d(interplanes, momentum=bn_mom)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv2d(interplanes, outplanes, kernel_size=1, padding=0, bias=True)
        self.scale_factor = scale_factor

    def forward(self, x):
        x = self.conv1(self.relu(self.bn1(x)))
        out = self.conv2(self.relu(self.bn2(x)))
        if self.scale_factor is not None:
            height = x.shape[-2] * self.scale_factor
            width = x.shape[-1] * self.scale_factor
            out = F.interpolate(out, size=[height, width], mode='bilinear', align_corners=algc)
        return out


class PAPPM(nn.Module):
    def __init__(self, inplanes, branch_planes, outplanes, BatchNorm=nn.BatchNorm2d):
        super().__init__()
        bn_mom = 0.1
        self.scale1 = nn.Sequential(nn.AvgPool2d(kernel_size=5, stride=2, padding=2), BatchNorm(inplanes, momentum=bn_mom),
                                    nn.ReLU(inplace=True), nn.Conv2d(inplanes, branch_planes, kernel_size=1, bias=False))
        self.scale2 = nn.Sequential(nn.AvgPool2d(kernel_size=9, stride=4, padding=4), BatchNorm(inplanes, momentum=bn_mom),
                                    nn.ReLU(inplace=True), nn.Conv2d(inplanes, branch_planes, kernel_size=1, bias=False))
        self.scale3 = nn.Sequential(nn.AvgPool2d(kernel_size=17, stride=8, padding=8), BatchNorm(inplanes, momentum=bn_mom),
                                    nn.ReLU(inplace=True), nn.Conv2d(inplanes, branch_planes, kernel_size=1, bias=False))
        self.scale4 = nn.Sequential(nn.AdaptiveAvgPool2d((1, 1)), BatchNorm(inplanes, momentum=bn_mom),
                                    nn.ReLU(inplace=True), nn.Conv2d(inplanes, branch_planes, kernel_size=1, bias=False))
        self.scale0 = nn.Sequential(BatchNorm(inplanes, momentum=bn_mom), nn.ReLU(inplace=True),
                                    nn.Conv2d(inplanes, branch_planes, kernel_size=1, bias=False))
        self.scale_process = nn.Sequential(BatchNorm(branch_planes * 4, momentum=bn_mom), nn.ReLU(inplace=True),
                                           nn.Conv2d(branch_planes * 4, branch_planes * 4, kernel_size=3, padding=1, groups=4, bias=False))
        self.compression = nn.Sequential(BatchNorm(branch_planes * 5, momentum=bn_mom), nn.ReLU(inplace=True),
                                         nn.Conv2d(branch_planes * 5, outplanes, kernel_size=1, bias=False))
        self.shortcut = nn.Sequential(BatchNorm(inplanes, momentum=bn_mom), nn.ReLU(inplace=True),
                                      nn.Conv2d(inplanes, outplanes, kernel_size=1, bias=False))

    def forward(self, x):
        width, height = x.shape[-1], x.shape[-2]
        x_ = self.scale0(x)
        scale_list = [
            F.interpolate(self.scale1(x), size=[height, width], mode='bilinear', align_corners=algc) + x_,
            F.interpolate(self.scale2(x), size=[height, width], mode='bilinear', align_corners=algc) + x_,
            F.interpolate(self.scale3(x), size=[height, width], mode='bilinear', align_corners=algc) + x_,
            F.interpolate(self.scale4(x), size=[height, width], mode='bilinear', align_corners=algc) + x_,
        ]
        scale_out = self.scale_process(torch.cat(scale_list, 1))
        return self.compression(torch.cat([x_, scale_out], 1)) + self.shortcut(x)


class PagFM(nn.Module):
    def __init__(self, in_channels, mid_channels, after_relu=False, with_channel=False, BatchNorm=nn.BatchNorm2d):
        super().__init__()
        self.with_channel = with_channel
        self.after_relu = after_relu
        self.f_x = nn.Sequential(nn.Conv2d(in_channels, mid_channels, kernel_size=1, bias=False), BatchNorm(mid_channels))
        self.f_y = nn.Sequential(nn.Conv2d(in_channels, mid_channels, kernel_size=1, bias=False), BatchNorm(mid_channels))
        if with_channel:
            self.up = nn.Sequential(nn.Conv2d(mid_channels, in_channels, kernel_size=1, bias=False), BatchNorm(in_channels))
        if after_relu:
            self.relu = nn.ReLU(inplace=True)

    def forward(self, x, y):
        input_size = x.size()
        if self.after_relu:
            y = self.relu(y)
            x = self.relu(x)
        y_q = self.f_y(y)
        y_q = F.interpolate(y_q, size=[input_size[2], input_size[3]], mode='bilinear', align_corners=False)
        x_k = self.f_x(x)
        if self.with_channel:
            sim_map = torch.sigmoid(self.up(x_k * y_q))
        else:
            sim_map = torch.sigmoid(torch.sum(x_k * y_q, dim=1).unsqueeze(1))
        y = F.interpolate(y, size=[input_size[2], input_size[3]], mode='bilinear', align_corners=False)
        return (1 - sim_map) * x + sim_map * y


class Light_Bag(nn.Module):
    def __init__(self, in_channels, out_channels, BatchNorm=nn.BatchNorm2d):
        super().__init__()
        self.conv_p = nn.Sequential(nn.Conv2d(in_channels, out_channels, kernel_size=1, bias=False), BatchNorm(out_channels))
        self.conv_i = nn.Sequential(nn.Conv2d(in_channels, out_channels, kernel_size=1, bias=False), BatchNorm(out_channels))

    def forward(self, p, i, d):
        edge_att = torch.sigmoid(d)
        p_add = self.conv_p((1 - edge_att) * i + p)
        i_add = self.conv_i(i + edge_att * p)
        return p_add + i_add


class PIDNet(nn.Module):
    def __init__(self, m=2, n=3, num_classes=2, planes=32, ppm_planes=96, head_planes=128, augment=True):
        super().__init__()
        self.augment = augment

        self.conv1 = nn.Sequential(
            nn.Conv2d(3, planes, kernel_size=3, stride=2, padding=1), BatchNorm2d(planes, momentum=bn_mom), nn.ReLU(inplace=True),
            nn.Conv2d(planes, planes, kernel_size=3, stride=2, padding=1), BatchNorm2d(planes, momentum=bn_mom), nn.ReLU(inplace=True),
        )
        self.relu = nn.ReLU(inplace=True)
        self.layer1 = self._make_layer(BasicBlock, planes, planes, m)
        self.layer2 = self._make_layer(BasicBlock, planes, planes * 2, m, stride=2)
        self.layer3 = self._make_layer(BasicBlock, planes * 2, planes * 4, n, stride=2)
        self.layer4 = self._make_layer(BasicBlock, planes * 4, planes * 8, n, stride=2)
        self.layer5 = self._make_layer(Bottleneck, planes * 8, planes * 8, 2, stride=2)

        self.compression3 = nn.Sequential(nn.Conv2d(planes * 4, planes * 2, kernel_size=1, bias=False), BatchNorm2d(planes * 2, momentum=bn_mom))
        self.compression4 = nn.Sequential(nn.Conv2d(planes * 8, planes * 2, kernel_size=1, bias=False), BatchNorm2d(planes * 2, momentum=bn_mom))
        self.pag3 = PagFM(planes * 2, planes)
        self.pag4 = PagFM(planes * 2, planes)

        self.layer3_ = self._make_layer(BasicBlock, planes * 2, planes * 2, m)
        self.layer4_ = self._make_layer(BasicBlock, planes * 2, planes * 2, m)
        self.layer5_ = self._make_layer(Bottleneck, planes * 2, planes * 2, 1)

        # m == 2 branch only (PIDNet-S/M share this; PIDNet-L uses DAPPM/Bag instead of PAPPM/Light_Bag).
        self.layer3_d = self._make_single_layer(BasicBlock, planes * 2, planes)
        self.layer4_d = self._make_layer(Bottleneck, planes, planes, 1)
        self.diff3 = nn.Sequential(nn.Conv2d(planes * 4, planes, kernel_size=3, padding=1, bias=False), BatchNorm2d(planes, momentum=bn_mom))
        self.diff4 = nn.Sequential(nn.Conv2d(planes * 8, planes * 2, kernel_size=3, padding=1, bias=False), BatchNorm2d(planes * 2, momentum=bn_mom))
        self.spp = PAPPM(planes * 16, ppm_planes, planes * 4)
        self.dfm = Light_Bag(planes * 4, planes * 4)

        self.layer5_d = self._make_layer(Bottleneck, planes * 2, planes * 2, 1)

        if self.augment:
            self.seghead_p = segmenthead(planes * 2, head_planes, num_classes)
            self.seghead_d = segmenthead(planes * 2, planes, 1)
        self.final_layer = segmenthead(planes * 4, head_planes, num_classes)

        for m_ in self.modules():
            if isinstance(m_, nn.Conv2d):
                nn.init.kaiming_normal_(m_.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m_, BatchNorm2d):
                nn.init.constant_(m_.weight, 1)
                nn.init.constant_(m_.bias, 0)

    def _make_layer(self, block, inplanes, planes, blocks, stride=1):
        downsample = None
        if stride != 1 or inplanes != planes * block.expansion:
            downsample = nn.Sequential(
                nn.Conv2d(inplanes, planes * block.expansion, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(planes * block.expansion, momentum=bn_mom))
        layers = [block(inplanes, planes, stride, downsample)]
        inplanes = planes * block.expansion
        for i in range(1, blocks):
            layers.append(block(inplanes, planes, stride=1, no_relu=(i == blocks - 1)))
        return nn.Sequential(*layers)

    def _make_single_layer(self, block, inplanes, planes, stride=1):
        downsample = None
        if stride != 1 or inplanes != planes * block.expansion:
            downsample = nn.Sequential(
                nn.Conv2d(inplanes, planes * block.expansion, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(planes * block.expansion, momentum=bn_mom))
        return block(inplanes, planes, stride, downsample, no_relu=True)

    def forward(self, x):
        width_output = x.shape[-1] // 8
        height_output = x.shape[-2] // 8

        x = self.conv1(x)
        x = self.layer1(x)
        x = self.relu(self.layer2(self.relu(x)))
        x_ = self.layer3_(x)
        x_d = self.layer3_d(x)

        x = self.relu(self.layer3(x))
        x_ = self.pag3(x_, self.compression3(x))
        x_d = x_d + F.interpolate(self.diff3(x), size=[height_output, width_output], mode='bilinear', align_corners=algc)
        if self.augment:
            temp_p = x_

        x = self.relu(self.layer4(x))
        x_ = self.layer4_(self.relu(x_))
        x_d = self.layer4_d(self.relu(x_d))
        x_ = self.pag4(x_, self.compression4(x))
        x_d = x_d + F.interpolate(self.diff4(x), size=[height_output, width_output], mode='bilinear', align_corners=algc)
        if self.augment:
            temp_d = x_d

        x_ = self.layer5_(self.relu(x_))
        x_d = self.layer5_d(self.relu(x_d))
        x = F.interpolate(self.spp(self.layer5(x)), size=[height_output, width_output], mode='bilinear', align_corners=algc)

        x_ = self.final_layer(self.dfm(x_, x, x_d))

        if self.augment:
            return [self.seghead_p(temp_p), x_, self.seghead_d(temp_d)]
        return x_


class OhemCrossEntropy(nn.Module):
    """utils/criterion.py -- only the LAST score in the list uses OHEM; earlier ones use plain CE."""

    def __init__(self, ignore_label=255, thres=0.9, min_kept=131072, balance_weights=(0.4, 1.0)):
        super().__init__()
        self.thresh = thres
        self.min_kept = max(1, min_kept)
        self.ignore_label = ignore_label
        self.balance_weights = balance_weights
        self.criterion = nn.CrossEntropyLoss(ignore_index=ignore_label, reduction='none')

    def _ce_forward(self, score, target):
        return self.criterion(score, target).mean()

    def _ohem_forward(self, score, target):
        pred = F.softmax(score, dim=1)
        pixel_losses = self.criterion(score, target).contiguous().view(-1)
        mask = target.contiguous().view(-1) != self.ignore_label

        tmp_target = target.clone()
        tmp_target[tmp_target == self.ignore_label] = 0
        pred = pred.gather(1, tmp_target.unsqueeze(1))
        pred, ind = pred.contiguous().view(-1)[mask].contiguous().sort()
        if pred.numel() == 0:
            return score.new_tensor(0.0)
        min_value = pred[min(self.min_kept, pred.numel() - 1)]
        threshold = max(min_value, self.thresh)

        pixel_losses = pixel_losses[mask][ind]
        pixel_losses = pixel_losses[pred < threshold]
        return pixel_losses.mean()

    def forward(self, scores, target):
        functions = [self._ce_forward] * (len(self.balance_weights) - 1) + [self._ohem_forward]
        return sum(w * fn(s, target) for w, s, fn in zip(self.balance_weights, scores, functions))


def weighted_bce(bd_pre, target):
    n, c, h, w = bd_pre.size()
    log_p = bd_pre.permute(0, 2, 3, 1).contiguous().view(1, -1)
    target_t = target.view(1, -1)
    pos_index = (target_t == 1)
    neg_index = (target_t == 0)
    weight = torch.zeros_like(log_p)
    pos_num = pos_index.sum()
    neg_num = neg_index.sum()
    sum_num = pos_num + neg_num
    weight[pos_index] = neg_num * 1.0 / sum_num
    weight[neg_index] = pos_num * 1.0 / sum_num
    return F.binary_cross_entropy_with_logits(log_p, target_t, weight, reduction='mean')


class BoundaryLoss(nn.Module):
    def __init__(self, coeff_bce=20.0):
        super().__init__()
        self.coeff_bce = coeff_bce

    def forward(self, bd_pre, bd_gt):
        return self.coeff_bce * weighted_bce(bd_pre, bd_gt)


def pidnet_loss(outputs, labels, bd_gt, sem_loss, bd_loss, ignore_label=255, sb_weight=1.0):
    """utils/utils.py's FullModel.forward, as a plain function. outputs = [aux_p, final, aux_d],
    already interpolated to label resolution."""
    loss_s = sem_loss(outputs[:-1], labels)
    loss_b = bd_loss(outputs[-1], bd_gt)
    filler = torch.ones_like(labels) * ignore_label
    bd_label = torch.where(torch.sigmoid(outputs[-1][:, 0, :, :]) > 0.8, labels, filler)
    loss_sb = sb_weight * sem_loss._ohem_forward(outputs[-2], bd_label)
    return loss_s + loss_b + loss_sb, {'loss_s': float(loss_s.item()), 'loss_b': float(loss_b.item()), 'loss_sb': float(loss_sb.item())}

In [ ]:
model = PIDNet(m=2, n=3, num_classes=2, planes=32, ppm_planes=96, head_planes=128, augment=True).to(DEVICE)

# See the markdown cell above: both the official per-file Google Drive links and the author's
# own posted fallback folder were checked directly (real HTTP requests, not just trusting the
# README's caution note) and both come back as Google Drive's "file/folder does not exist" page.
# No faithful pretrained checkpoint is available -- training starts from the paper's own
# from-scratch Kaiming initialization scheme, already applied inside PIDNet.__init__.
pretrained_loaded = False
wandb.config.update({'pretrained_backbone_loaded': pretrained_loaded})
print('Trainable parameters:', sum(p.numel() for p in model.parameters() if p.requires_grad))

In [ ]:
@torch.inference_mode()
def evaluate(loader, measure_inference=False):
    model.eval()
    true_positive = false_positive = false_negative = 0
    inference_seconds = 0.0
    image_count = 0

    for batch in loader:
        pixel_values = batch['pixel_values'].to(DEVICE, non_blocking=True)
        labels = batch['labels'].to(DEVICE, non_blocking=True)

        if measure_inference:
            torch.cuda.synchronize()
            start = time.perf_counter()
        # augment=True is fixed at construction (not tied to .eval()), so this always returns
        # [aux_p, final, aux_d] at 1/8 resolution -- only the final output (index 1) matters here.
        outputs = model(pixel_values)
        final_logits = F.interpolate(outputs[1], size=labels.shape[-2:], mode='bilinear', align_corners=algc)
        if measure_inference:
            torch.cuda.synchronize()
            inference_seconds += time.perf_counter() - start

        predictions = final_logits.argmax(dim=1)
        true_positive += int(((predictions == 1) & (labels == 1)).sum().item())
        false_positive += int(((predictions == 1) & (labels == 0)).sum().item())
        false_negative += int(((predictions == 0) & (labels == 1)).sum().item())
        image_count += labels.shape[0]

    precision = true_positive / max(true_positive + false_positive, 1)
    recall = true_positive / max(true_positive + false_negative, 1)
    iou = true_positive / max(true_positive + false_positive + false_negative, 1)
    dice = 2 * true_positive / max(2 * true_positive + false_positive + false_negative, 1)

    return {
        'precision': precision,
        'recall': recall,
        'iou': iou,
        'dice': dice,
        # Pixel-level FPs per image, not per-instance -- this model has no discrete
        # "detections" to count false positives over, so it's normalized the honest
        # way: how many false-positive pixels on average per image in this split.
        'fp_per_image': false_positive / max(image_count, 1),
        'inference_time_ms_per_image': 1000 * inference_seconds / max(image_count, 1),
        'images': image_count,
    }


sem_loss = OhemCrossEntropy(ignore_label=255, thres=0.9, min_kept=131072, balance_weights=(0.4, 1.0))
bd_loss = BoundaryLoss(coeff_bce=20.0)

optimizer = AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scaler = torch.amp.GradScaler('cuda', enabled=True)

history = []
best_dice = -1.0
best_epoch = -1
epochs_without_improvement = 0

for epoch in range(1, MAX_EPOCHS + 1):
    model.train()
    running_loss = 0.0

    for batch in train_loader:
        pixel_values = batch['pixel_values'].to(DEVICE, non_blocking=True)
        labels = batch['labels'].to(DEVICE, non_blocking=True)
        bd_gt = batch['bd_gt'].to(DEVICE, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type='cuda', dtype=torch.float16):
            outputs = model(pixel_values)
            outputs = [F.interpolate(o, size=labels.shape[-2:], mode='bilinear', align_corners=algc) for o in outputs]
            loss, loss_parts = pidnet_loss(outputs, labels, bd_gt, sem_loss, bd_loss)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        running_loss += float(loss.item())

    val_metrics = evaluate(val_loader)
    row = {'epoch': epoch, 'train_loss': running_loss / len(train_loader), **val_metrics}
    history.append(row)
    print(f"Epoch {epoch:02d}/{MAX_EPOCHS} | loss={row['train_loss']:.4f} | val Dice={row['dice']:.4f} | val IoU={row['iou']:.4f} | val Recall={row['recall']:.4f}")
    wandb.log(row, step=epoch)

    if row['dice'] > best_dice:
        best_dice = row['dice']
        best_epoch = epoch
        epochs_without_improvement = 0
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_metrics': val_metrics,
        }, RUN_DIR / 'best_model.pt')
    else:
        epochs_without_improvement += 1

    pd.DataFrame(history).to_csv(RUN_DIR / 'training_history.csv', index=False)
    if epochs_without_improvement >= PATIENCE:
        print(f'Early stopping at epoch {epoch}; best validation Dice was {best_dice:.4f} at epoch {best_epoch}.')
        break

wandb.summary['best_epoch'] = best_epoch
wandb.summary['best_validation_dice'] = best_dice
print('Best epoch:', best_epoch, 'Best validation Dice:', best_dice)

In [ ]:
# Load the best validation-Dice checkpoint and evaluate every fixed test subset.
checkpoint = torch.load(RUN_DIR / 'best_model.pt', map_location=DEVICE, weights_only=False)
model.load_state_dict(checkpoint['model_state_dict'])

test_rows = []
for split_name, split_samples in test_sets.items():
    metrics = evaluate(make_loader(split_samples), measure_inference=True)
    test_rows.append({'split': split_name, **metrics})
    print(split_name, metrics)
    wandb.log({f'test_{split_name}_{key}': value for key, value in metrics.items()})

test_df = pd.DataFrame(test_rows)
display(test_df)

FINAL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
shutil.copy2(RUN_DIR / 'best_model.pt', FINAL_OUTPUT_DIR / 'best_model.pt')
shutil.copy2(RUN_DIR / 'training_history.csv', FINAL_OUTPUT_DIR / 'training_history.csv')
# Long-format, one row per split -- same shape as D-FINE's evaluation_metrics.csv.
test_df.to_csv(FINAL_OUTPUT_DIR / 'evaluation_metrics.csv', index=False)

by_split = {row['split']: row for row in test_rows}
overall = by_split['overall']

# Wide-format, one row per experiment -- matches the D-FINE/detection summary.csv schema
# exactly so this row can drop straight into the same combined results table. mAP columns
# are intentionally left blank: mAP is an instance-level metric and doesn't have a valid
# equivalent for a dense per-pixel segmenter like this one. Dice/IoU are kept as the honest
# segmentation-quality headline metric instead of a fabricated mAP.
summary_row = {
    'Experiment': RUN_NAME,
    'Model': MODEL_LABEL,
    'Batch': BATCH_SIZE,
    'Epochs': best_epoch,
    'mAP50': float('nan'),
    'mAP50_95': float('nan'),
    'Precision': overall['precision'],
    'Recall': overall['recall'],
    'mAP50_Small': float('nan'),
    'mAP50_Medium': float('nan'),
    'mAP50_Large': float('nan'),
    'Recall_Small': by_split['small']['recall'],
    'Recall_Medium': by_split['medium']['recall'],
    'Recall_Large': by_split['large']['recall'],
    'Inference_Time_ms': overall['inference_time_ms_per_image'],
    'FP_per_Image': overall['fp_per_image'],
    'Dice': overall['dice'],
    'IoU': overall['iou'],
    'Notes': f'{MODEL_LABEL}, vendored 3-branch (P/I/D) architecture with faithful OHEM+boundary-BCE+boundary-masked-aux loss (no framework dependency), pretrained_backbone_loaded={pretrained_loaded} (official weights verified unavailable, trained from scratch), binary defect segmentation, fixed 640 split. mAP intentionally blank -- not a valid metric for dense semantic segmentation.',
}
summary_df = pd.DataFrame([summary_row])
display(summary_df)
summary_df.to_csv(FINAL_OUTPUT_DIR / 'summary.csv', index=False)

metadata = {
    'experiment': RUN_NAME,
    'model': MODEL_LABEL,
    'task': 'binary semantic defect segmentation',
    'pretrained_backbone_loaded': pretrained_loaded,
    'image_size': IMG_SIZE,
    'batch_size': BATCH_SIZE,
    'max_epochs': MAX_EPOCHS,
    'early_stopping_patience': PATIENCE,
    'best_epoch': best_epoch,
    'best_validation_dice': best_dice,
    'split_counts': {
        'train': len(train_samples), 'val': len(val_samples),
        'test': len(test_samples), 'test_small': len(test_sets['small']),
        'test_medium': len(test_sets['medium']), 'test_large': len(test_sets['large']),
    },
}
(FINAL_OUTPUT_DIR / 'run_metadata.json').write_text(json.dumps(metadata, indent=2))
print('Saved model and all metrics to:', FINAL_OUTPUT_DIR)

wandb.finish()